# 01 Agroforestry: planted-silvoarable first pass, Haiti

Computes CIAT's **partial terrain-and-soil index** (slope 50%, soil organic carbon 25%, pH 25%, with a host gate)
over the AOI, runs the **ordinal simulation** next to it, and compares both on maps.

Edit **only the parameters cell**, then run all. Every run saves its configuration, fingerprint, maps and a
manifest to `runs/<fingerprint>/`.

- `MODE = "exploratory"`: the legal-exclusion mask is **not applied** and open decisions stay open, so the maps can be
  compared. Every map and the manifest say so. Not policy-conformant.
- `MODE = "policy"`: refuses to run until every decision is recorded.
- `DATA_SOURCE = "synthetic"` tests the pipeline offline; its results are never Haiti.

In [ ]:
# ===== PARAMETERS: edit only this cell =====
CONFIG_PATH = "../config/runs/agroforestry_planted_first_pass.toml"   # base settings
MODE = "exploratory"                      # "exploratory" or "policy"
DATA_SOURCE = "gee"                       # "gee" or "synthetic" (offline pipeline test)
EE_PROJECT = ""                           # your Earth Engine project id (not committed anywhere)

# AOI (area of interest)
AOI_NAME = "Haiti (approximate box)"
AOI_BBOX = (-74.5, 18.0, -71.6, 20.1)     # minx, miny, maxx, maxy in EPSG:4326
AOI_SOURCE = ""                           # official boundary and where it comes from, for example "GADM 4.1"

# Grid
RESOLUTION_M = 5000                       # 5 km is 47 x 65 cells; finer grids take much longer in Earth Engine
SCENARIO = "baseline"

# Target and decisions (leave empty what is not decided)
ADMIN_COUNTRY = "HTI"
APPLICABILITY_DECISION = ""               # reference for using the India-derived curves in the target
SOIL_ADOPTED = False                      # True only with a decision reference below
SOIL_DECISION_REFERENCE = ""
LEGAL_MASK_SPEC = ""                      # assessed legal-exclusion rule and its coverage

# Ordinal simulation
N_DRAWS = 5000
SEED = 20261103                           # same seed, same result
SCALE_MODE = "independent"                # "independent" or "shared"
WEIGHT_CONCENTRATION = 0                  # 0 keeps CIAT's weights fixed; for example 20 lets them vary

# Where a layer is looked for, in this order (the first pass reads Earth Engine only, for now)
DATA_ORDER = ("local", "hf", "gee")
HF_REPO = "dagudelo30/nbs-ruralscan-data"

## 1. Run configuration

In [ ]:
from pathlib import Path

from nbs_runtime.runconfig import RunConfig

OVERRIDES = {
    "run": {"mode": MODE},
    "aoi": {"name": AOI_NAME, "bbox": list(AOI_BBOX), "source": AOI_SOURCE},
    "grid": {"resolution_m": RESOLUTION_M, "scenario": SCENARIO},
    "target": {"country": ADMIN_COUNTRY, "applicability_decision": APPLICABILITY_DECISION},
    "soil": {"adopted": SOIL_ADOPTED, "decision_reference": SOIL_DECISION_REFERENCE},
    "masks": {"legal_mask_spec": LEGAL_MASK_SPEC},
    "simulation": {"n_draws": N_DRAWS, "seed": SEED, "scale_mode": SCALE_MODE,
                   "weight_concentration": WEIGHT_CONCENTRATION},
    "data": {"order": list(DATA_ORDER), "hf_repo": HF_REPO},
}
cfg = RunConfig.load(CONFIG_PATH, OVERRIDES)

OUT_DIR = Path("../runs") / cfg.fingerprint()[:10]
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "run_config.json").write_text(cfg.to_json(), encoding="utf-8")
print("run folder:", OUT_DIR, "| config fingerprint:", cfg.fingerprint()[:12])

## 2. Plan: what is ready and what is on hold

In [ ]:
from nbs_runtime.plan import plan

p = plan(cfg)
aoi = cfg.aoi
print(f"CIAT package {p.package_version} @ {p.package_ref[:8]} | policy {p.policy_id} | mode {cfg.run.mode}")
print(f"AOI: {aoi.name} | box area {aoi.area_km2():,.0f} km2 | "
      f"about {aoi.estimated_cells(cfg.grid.resolution_m):,} cells at {cfg.grid.resolution_m} m")
print("READY" if p.ready else "ON HOLD")
for h in p.holds:
    print("  hold:", h)
for w in p.warnings:
    print("  note:", w)

if cfg.run.mode == "policy" and not p.ready:
    raise SystemExit("policy mode needs every decision recorded; resolve the holds above, or use MODE = 'exploratory'")

## 3. Layers

Earth Engine, following CIAT's `DATASET_INPUTS.md`: slope is derived from SRTM at its native ~30 m scale and **then**
averaged to the grid; soil is SoilGrids at 0-5, 5-15 and 15-30 cm, converted once and weighted by layer thickness
(`planted_topsoil_0_30_v1`, a scoping proposal CIAT has not selected); the host gate uses WorldCover class shares.
Missing values stay unknown: nothing is filled.

In [ ]:
import numpy as np

from nbs_runtime.contract import load_package
from nbs_runtime.data import gee
from nbs_runtime.data.synthetic import synthetic_layers
from nbs_runtime.grid import Grid
from nbs_runtime.rasters import gate_class_groups

pkg = load_package()
policy = pkg.policy(cfg.run.policy_id)
grid = Grid.from_aoi(cfg.aoi, cfg.grid.resolution_m)
groups = gate_class_groups(pkg.gate_params(policy))
print("grid:", grid.shape, "rows x columns | host classes:", groups)

if DATA_SOURCE == "synthetic":
    layers = synthetic_layers(grid, seed=cfg.simulation.seed)
else:
    gee.init_ee(EE_PROJECT)
    layers = gee.load_layers(grid, groups)

def describe(name, a):
    ok = a[~np.isnan(a)]
    rng = f"{ok.min():.3g} to {ok.max():.3g}, median {np.median(ok):.3g}" if ok.size else "no data"
    print(f"  {name:<22} {rng} | unknown {100 * np.isnan(a).mean():.1f}%")

for n in layers.notes:
    print("note:", n)
describe("slope (deg)", layers.slope_deg)
describe("SOC 0-30 cm (g/kg)", layers.soc_g_kg)
describe("pH 0-30 cm", layers.ph)
describe("eligible host share", layers.eligible_fraction)
describe("unknown host share", layers.unknown_fraction)

## 4. CIAT's calculation

Scores from CIAT's functions, host gate, weighted sum 50/25/25, unknown propagated, no renormalisation.

In [ ]:
from nbs_runtime.firstpass import run_first_pass
from nbs_runtime.simulate import build_level_space, simulate

space = build_level_space(pkg, policy)
fp = run_first_pass(space, layers, cfg.run.mode)

total = fp.status.size
print(f"cells: {total:,}")
for name, n in fp.status_counts().items():
    print(f"  {name:<38}{n:>8,}  {100 * n / total:5.1f}%")
if fp.scored.any():
    print(f"CIAT index over scored cells: min {np.nanmin(fp.pete):.3f}, median {np.nanmedian(fp.pete):.3f}, max {np.nanmax(fp.pete):.3f}")

## 5. Ordinal simulation

The order of the classes and the class of each cell stay fixed; the numbers given to the classes are redrawn many
times in increasing order. Each cell's index becomes a median and a 5 to 95% range, next to CIAT's own value.

In [ ]:
from nbs_runtime.compare import compare

s = cfg.simulation
res = simulate(space, s.n_draws, s.seed, s.scale_mode, s.prior, s.weight_concentration)
comp = compare(fp, res)

st = comp.stats
if st["n_scored"]:
    print(f"scored cells: {st['n_scored']:,} of {st['n_cells']:,}")
    print(f"rank agreement, CIAT vs simulation median (Spearman): {st['spearman_pete_vs_median']:.3f}")
    print(f"share of cells in CIAT's top third: {100 * st['share_pete_top_third']:.1f}%")
    print(f"agreement on top-third membership: {100 * st['top_third_agreement']:.1f}%")
    print(f"CIAT top-third cells that are fragile (chance of top third below 50%): {100 * st['share_of_pete_top_third_that_is_fragile']:.1f}%")
    print(f"cells with an uncertain top third (chance between 20% and 80%): {100 * st['share_uncertain_top_third']:.1f}%")
    print(f"width of the 5-95% range: median {st['width_median']:.2f}, 90th percentile {st['width_p90']:.2f}")
    print(f"cells where CIAT's value is above the simulation's 95th percentile: {100 * st['share_pete_above_p95']:.1f}%")
else:
    print("no scored cells: nothing to compare (policy mode, or no eligible host in the AOI)")

## 6. Maps

In [ ]:
import matplotlib.pyplot as plt

from nbs_runtime.viz import plot_comparison

flags = []
if cfg.run.mode == "exploratory":
    flags.append("EXPLORATORY: legal mask not applied, decisions open")
if layers.synthetic:
    flags.append("SYNTHETIC DATA: pipeline test, not Haiti")
if not cfg.aoi.source:
    flags.append("AOI is a placeholder box")
title = f"{'SYNTHETIC TEST: ' if layers.synthetic else ''}{cfg.aoi.name}: CIAT's index vs ordinal simulation ({cfg.simulation.scale_mode} scales, {cfg.simulation.n_draws:,} draws)"
note = " | ".join(flags) + f" | {cfg.grid.resolution_m} m grid | config {cfg.fingerprint()[:10]}"
fig = plot_comparison(fp, comp, grid, title, note, OUT_DIR / "maps.png")
plt.show()

## 7. Save this run

In [ ]:
import json

np.savez_compressed(
    OUT_DIR / "layers_and_results.npz",
    slope_deg=layers.slope_deg, soc_g_kg=layers.soc_g_kg, ph=layers.ph,
    eligible_fraction=layers.eligible_fraction, unknown_fraction=layers.unknown_fraction,
    status=fp.status, ciat_index=fp.pete, combo_id=fp.combo_id,
    sim_median=comp.median, sim_p05=comp.p05, sim_p95=comp.p95, p_top_third=comp.p_top,
    depth_range=fp.depth_range if fp.depth_range is not None else np.array([]),
    grid_bbox=np.array(grid.bbox), grid_resolution_deg=grid.resolution_deg,
)
manifest = {
    "config_fingerprint": cfg.fingerprint(),
    "aoi_fingerprint": cfg.aoi.fingerprint(),
    "mode": cfg.run.mode,
    "exploratory": cfg.run.mode == "exploratory",
    "data_source": DATA_SOURCE,
    "synthetic": layers.synthetic,
    "ciat_package": {"version": pkg.version, "ref": pkg.pin["ref"], "source_commit": pkg.pin["package_source_commit"]},
    "plan": {"ready": p.ready, "holds": p.holds, "warnings": p.warnings},
    "layer_notes": layers.notes,
    "grid": {"shape": list(grid.shape), "resolution_m": grid.resolution_m, "bbox": list(grid.bbox)},
    "status_counts": fp.status_counts(),
    "comparison": {k: (None if isinstance(v, float) and np.isnan(v) else v) for k, v in comp.stats.items()},
    "simulation": res.metadata(),
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("saved:", sorted(x.name for x in OUT_DIR.iterdir()))

## Reading the maps

- **CIAT index** and **simulation median** use the same 0-1 scale, but the simulation draws its scores with no
  prior information, so its median sits below CIAT's convention (0.25, 0.5, 0.75, 0.9). Compare the ranking, not the level.
- **5-95% width** shows where the result depends most on the numbers given to the classes.
- **Chance of the top third** is the share of scenarios in which the cell is among the best third of the scored cells.
- **Pixel state** keeps excluded, unknown and scored cells apart. Unknown is never turned into zero.

## Next

- Record the open decisions in the parameters cell as they are made (AOI boundary, applicability, soil, legal mask).
- Switch to `MODE = "policy"` once they are recorded.